In [1]:
import joblib
import pandas as pd
import numpy as np
import time

t0 = time.time()
preprocessor = joblib.load("../models/preprocessor.joblib")
print(f"[{time.time()-t0:.0f}с] Препроцессор загружен")

model = joblib.load("../models/forest_pha.joblib")
print(f"[{time.time()-t0:.0f}с] Модель загружена")

df_part = pd.read_csv("../data/raw/dataset.csv", nrows=200_000, low_memory=False)
print(f"[{time.time()-t0:.0f}с] Данные прочитаны")

feature_cols = [c for c in df_part.columns if c not in
                ("id", "spkid", "full_name", "pdes", "name", "prefix",
                 "orbit_id", "equinox", "moid_ld", "epoch_mjd", "epoch_cal",
                 "per_y", "tp_cal", "diameter_sigma", "diameter", "pha")]
typical = df_part[feature_cols].median(numeric_only=True).to_dict()

print(f"[{time.time()-t0:.0f}с] ГОТОВО ✅ | Признаков:", len(feature_cols))

[2с] Препроцессор загружен
[3с] Модель загружена
[5с] Данные прочитаны
[5с] ГОТОВО ✅ | Признаков: 29


In [5]:
print("🛡️ ASTEROID GUARD — введите параметры астероида 🪨\n")
print("Подсказка: просто нажмите Enter — подставится значение по умолчанию\n")

def smart_float(prompt, default):
    raw = input(prompt).strip().lower().replace(",", ".")
    # вытаскиваем число из чего угодно: "moid=1.5" -> "1.5"
    num = "".join(ch for ch in raw if ch.isdigit() or ch == ".")
    return float(num) if num else default

moid = smart_float("MOID, а.е. (0.05 = очень близко): ", 0.05)
H    = smart_float("Яркость H (20 = средний): ", 20.0)
e    = smart_float("Эксцентриситет e (0.3): ", 0.3)
a    = smart_float("Большая полуось a (2.0): ", 2.0)
q    = smart_float("Перигелий q (1.0): ", 1.0)
neo  = input("NEO близкоземный? Y/N (Y): ").strip().upper() or "Y"
cls  = input("Класс орбиты APO/ATE/AMO/MBA (APO): ").strip().upper() or "APO"

asteroid = dict(typical)
asteroid.update({"moid": moid, "H": H, "e": e, "a": a, "q": q,
                 "neo": neo, "class": cls})
row = pd.DataFrame([asteroid])[feature_cols]

proba = model.predict_proba(preprocessor.transform(row))[0, 1]

print("\n" + "=" * 50)
print("        🛡️  ВЕРДИКТ МОДЕЛИ  🛡️")
print("=" * 50)
print(f"  MOID {moid:.3f} а.е. = {moid*215:.0f} лунных расстояний")
print("-" * 50)
if proba >= 0.5:
    print(f"  ⚠️  ОПАСЕН — вероятность {proba*100:.1f}%")
    print("  Рекомендация: приоритетное наблюдение")
else:
    print(f"  ✅ НЕ ОПАСЕН — вероятность {proba*100:.1f}%")
print("=" * 50)

🛡️ ASTEROID GUARD — введите параметры астероида 🪨

Подсказка: просто нажмите Enter — подставится значение по умолчанию


        🛡️  ВЕРДИКТ МОДЕЛИ  🛡️
  MOID 1.500 а.е. = 322 лунных расстояний
--------------------------------------------------
  ✅ НЕ ОПАСЕН — вероятность 20.0%
